<a href="https://colab.research.google.com/github/ArrenOnom/Tropical-Basin-Erosion-Modeling-Pipeline/blob/main/Gully_Erosion_Susceptibility_Mapping_Idemili_Awka_ML_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Integrating Machine Learning and Geomatics in Modelling Gully Erosion Susceptibility in Awka and Idemili Areas of Anambra State, Nigeria**


**Author:** Nelson Onyebuchi Nwobi

**Date:** April 2026

**License:** MIT License


**Project Overview**

This notebook contains the statistical analysis and machine learning pipeline for modeling gully erosion susceptibility in the Idemili and Awka regions of Anambra State, Nigeria. By integrating multi-source geospatial data (SRTM DEM, SoilGrids) with 3D-verified gully inventories, this study compares the performance of Random Forest (RF) and Binary Logistic Regression (BLR) algorithms.

**Key Objectives:**

1.	Data Preprocessing: Cleaning and normalizing topographic and lithological covariates.
2.	Model Training: Implementing RF and BLR models to identify primary erosion drivers.
3.	Validation: Assessing model accuracy using Area Under the ROC Curve (AUC), Precision, and Recall.
4.	Susceptibility Mapping: Generating high-resolution GSI (Gully Susceptibility Index) outputs.

**Dependencies:**

To run this notebook, ensure you have the following Python libraries installed:

•	pandas, numpy (Data Handling)

•	scikit-learn (Machine Learning)

•	matplotlib, seaborn (Visualization)

•	rasterio, geopandas (Geospatial Processing)





In [1]:
# ==============================================================================
# REPOSITORY SETUP: AUTOMATIC CLONING & PATH MAPPING
# ==============================================================================
import os

# 1. YOUR SPECIFIC REPOSITORY
REPO_URL = "https://github.com/ArrenOnom/Tropical-Basin-Erosion-Modeling-Pipeline.git"
REPO_NAME = "Tropical-Basin-Erosion-Modeling-Pipeline"

# 2. CLONE LOGIC
if not os.path.exists(REPO_NAME):
    print(f"--- Cloning {REPO_NAME} from GitHub ---")
    !git clone {REPO_URL}
else:
    print(f"--- {REPO_NAME} exists. Updating... ---")
    %cd {REPO_NAME}
    !git pull
    %cd ..

# 3. SET BASE DIRECTORY
# This variable 'BASE_DIR' is now the "Anchor" for all your other scripts.
BASE_DIR = os.path.join(os.getcwd(), REPO_NAME)
print(f"\n✅ SUCCESS: Base Directory is {BASE_DIR}")

--- Cloning Tropical-Basin-Erosion-Modeling-Pipeline from GitHub ---
Cloning into 'Tropical-Basin-Erosion-Modeling-Pipeline'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 118 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 23.53 MiB | 18.03 MiB/s, done.
Resolving deltas: 100% (21/21), done.

✅ SUCCESS: Base Directory is /content/Tropical-Basin-Erosion-Modeling-Pipeline


In [ ]:
# ==============================================================================
# SECTION 4.2 – DESCRIPTIVE STATISTICS (MANUSCRIPT REVISION)
# 1200 DPI | BOLD FONTS | AUTOMATED FOLDER MAPPING | DATA CLEANING
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# ------------------------------------------------------------------------------
# 1. DYNAMIC PATH CONFIGURATION
# ------------------------------------------------------------------------------
# Get the base directory (Assuming notebook is in 'Python_Analysis' folder)
BASE_DIR = os.path.dirname(os.getcwd())

# Define input and output paths based on GitHub structure
PATH_AWKA_DATA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI_DATA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Output Folders
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')

# Ensure output directories exist
for folder in [OUT_AWKA, OUT_IDEMILI]:
    if not os.path.exists(folder):
        os.makedirs(folder)

# ------------------------------------------------------------------------------
# 2. DATA LOADING & ROBUST CLEANING
# ------------------------------------------------------------------------------
print("--- Loading and Cleaning Datasets ---")
idemili = pd.read_csv(PATH_IDEMILI_DATA)
awka = pd.read_csv(PATH_AWKA_DATA)

# Variables for SoilGrids cleaning and final analysis
soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
analysis_vars = topo_vars + soil_vars

# Remove missing/null values and SoilGrids noise (-9999)
for df_temp in [idemili, awka]:
    for col in soil_vars:
        # Replace negative/placeholder values with NaN
        df_temp.loc[df_temp[col] < 0, col] = np.nan
    df_temp.dropna(subset=analysis_vars, inplace=True)

# Assign labels for comparative analysis
idemili['Region'] = 'Idemili (Gully-Prone)'
awka['Region'] = 'Awka (Stable Basin)'

# Combine for descriptive aggregation
df_combined = pd.concat([idemili, awka], ignore_index=True)

# Calculate Statistics
stats = df_combined.groupby('Region')[analysis_vars].agg(['mean', 'std']).T.reset_index()
stats.columns = ['Variable', 'Stat', 'Awka_Val', 'Idemili_Val']

means = stats[stats['Stat'] == 'mean'].copy()
stds = stats[stats['Stat'] == 'std'].copy()

# ------------------------------------------------------------------------------
# 3. GLOBAL PLOT STYLING (MANUSCRIPT STANDARDS)
# ------------------------------------------------------------------------------
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.linewidth': 2.5,        # Heavy border for printing
    'xtick.major.width': 2,
    'ytick.major.width': 2,
    'legend.frameon': True,
    'legend.edgecolor': 'black',
    'font.family': 'sans-serif'
})

# ------------------------------------------------------------------------------
# 4. DATA VISUALIZATION (1200 DPI ULTRA HD)
# ------------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(16, 12))
y_pos = np.arange(len(analysis_vars))
bar_height = 0.4

# Plotting with High-Contrast Color Palette
# Awka: Dark Charcoal (#2c3e50) | Idemili: International Orange (#d35400)
ax.barh(y_pos - bar_height/2, means['Awka_Val'], xerr=stds['Awka_Val'],
        height=bar_height, label='Awka (Control Area)', color='#2c3e50',
        capsize=12, error_kw={'elinewidth': 3, 'ecolor': '#e74c3c'})

ax.barh(y_pos + bar_height/2, means['Idemili_Val'], xerr=stds['Idemili_Val'],
        height=bar_height, label='Idemili (Gully-Prone)', color='#d35400',
        capsize=12, error_kw={'elinewidth': 3, 'ecolor': 'black'})

# Axis Labelling and Bold Formatting
ax.set_yticks(y_pos)
ax.set_yticklabels(analysis_vars, fontsize=16, fontweight='bold')
ax.set_xlabel('Comparative Mean Values (with Standard Deviation Bars)',
              fontsize=18, fontweight='bold', labelpad=20)
ax.set_title('Figure 4.2: Statistical Distribution of Environmental Covariates',
             fontsize=20, fontweight='bold', pad=25)

# Legend Configuration
legend = ax.legend(loc='lower right', fontsize=15, borderpad=1.5, shadow=True)
legend.get_frame().set_linewidth(2.0)

# Final Polish
ax.grid(axis='x', linestyle='--', alpha=0.4, linewidth=1.5)
sns.despine(trim=False)
plt.tight_layout()

# ------------------------------------------------------------------------------
# 5. EXPORTING RESULTS
# ------------------------------------------------------------------------------
# Saving High-Resolution Image for Manuscript
fig_name = 'Fig_4_2_Descriptive_Analysis_1200DPI.png'
plt.savefig(os.path.join(OUT_IDEMILI, fig_name), dpi=1200, bbox_inches='tight')
plt.savefig(os.path.join(OUT_AWKA, fig_name), dpi=1200, bbox_inches='tight')

# Saving Clean Statistical Table for MS Word/Excel
table_name = 'Table_4_2_Statistical_Summary.csv'
final_table = means[['Variable', 'Awka_Val', 'Idemili_Val']].copy()
final_table.to_csv(os.path.join(OUT_IDEMILI, table_name), index=False)
final_table.to_csv(os.path.join(OUT_AWKA, table_name), index=False)

print(f"\n✅ SUCCESS: Ultra-HD Figure saved to Outputs subfolders.")
print(f"✅ SUCCESS: Statistical Table saved as: {table_name}")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.3A – CORRELATION MATRIX & MULTICOLLINEARITY (VIF)
# 1200 DPI | BOLD FONTS | MANUSCRIPT STANDARDS | PATH-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import os

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())

# Inputs
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Outputs (Saving to both folders for redundancy in the repository)
OUT_FOLDERS = [
    os.path.join(BASE_DIR, 'Outputs', 'IDEMILI'),
    os.path.join(BASE_DIR, 'Outputs', 'AWKA')
]

for folder in OUT_FOLDERS:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. DATA AGGREGATION & CLEANING
# ------------------------------------------------------------------------------
print("--- Initializing Correlation and VIF Analysis ---")
df_idm = pd.read_csv(PATH_IDEMILI)
df_awk = pd.read_csv(PATH_AWKA)

# Standardize variable lists
soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
vars_used = topo_vars + soil_vars

# Clean SoilGrids artifacts (-9999)
df_all = pd.concat([df_idm, df_awk], ignore_index=True)
for col in soil_vars:
    df_all.loc[df_all[col] < 0, col] = np.nan

df_clean = df_all.dropna(subset=vars_used)
data_matrix = df_clean[vars_used]

# ------------------------------------------------------------------------------
# 3. PEARSON CORRELATION HEATMAP (1200 DPI)
# ------------------------------------------------------------------------------
corr_matrix = data_matrix.corr(method='pearson')

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2})
plt.figure(figsize=(14, 11))

# Plotting High-Contrast Heatmap
sns.heatmap(
    corr_matrix, cmap='RdBu_r', center=0, annot=True, fmt=".2f",
    annot_kws={'size': 13, 'weight': 'bold', 'color': 'black'},
    linewidths=2, linecolor='white',
    cbar_kws={"shrink": 0.8, "label": "Pearson Correlation Coefficient (r)"}
)

plt.title('Figure 4.3a: Inter-Variable Correlation Matrix', fontsize=18, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()

# Save Heatmap to both folders
for folder in OUT_FOLDERS:
    plt.savefig(os.path.join(folder, 'Fig_4_3a_Correlation_Heatmap_1200DPI.png'), dpi=1200, bbox_inches='tight')

# ------------------------------------------------------------------------------
# 4. MULTICOLLINEARITY ANALYSIS (VIF)
# ------------------------------------------------------------------------------
# Add a constant for VIF calculation
X = sm.add_constant(data_matrix)
vif_df = pd.DataFrame()
vif_df['Variable'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Remove the constant row and round for clean presentation
vif_results = vif_df[vif_df['Variable'] != 'const'].copy().round(3)

# --- VIF Visualization (High Resolution)
plt.figure(figsize=(11, 7))
vif_bar = sns.barplot(
    data=vif_results.sort_values('VIF', ascending=False),
    x='VIF', y='Variable',
    palette='magma', edgecolor='black', linewidth=2
)

# Threshold Line (Academic Standard = 5.0)
plt.axvline(5, color='#e74c3c', linestyle='--', linewidth=3, label='Tolerance Threshold (VIF = 5)')

plt.title('Figure 4.3b: Multicollinearity Assessment (VIF Scores)', fontsize=18, fontweight='bold', pad=20)
plt.xlabel('Variance Inflation Factor (VIF)', fontsize=14, fontweight='bold')
plt.ylabel('')
plt.xticks(fontsize=12, fontweight='bold')
plt.yticks(fontsize=12, fontweight='bold')
plt.legend(loc='lower right', fontsize=12, frameon=True, edgecolor='black')
sns.despine()
plt.tight_layout()

# Save VIF Plot and Tables
for folder in OUT_FOLDERS:
    plt.savefig(os.path.join(folder, 'Fig_4_3b_VIF_Analysis_1200DPI.png'), dpi=1200, bbox_inches='tight')
    vif_results.to_csv(os.path.join(folder, 'Table_4_3b_VIF_Scores.csv'), index=False)
    corr_matrix.to_csv(os.path.join(folder, 'Table_4_3a_Correlation_Matrix.csv'))

print(f"\n✅ SUCCESS: Correlation Heatmap and VIF Analysis saved at 1200 DPI.")
print(f"✅ DATA STATUS: Multicollinearity checked. Threshold is VIF < 5.0.")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.3B – PANEL FIGURE 6: MULTICOLLINEARITY DIAGNOSIS (A & B)
# 1200 DPI | MANUSCRIPT PANEL LABELS | HIGH CONTRAST | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

# Ensure directories exist
for folder in [OUT_IDEMILI, OUT_AWKA]:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. GLOBAL VISUAL PARAMETERS
# ------------------------------------------------------------------------------
plt.rcParams.update({
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.linewidth': 2.5,
    'font.family': 'sans-serif'
})

# ------------------------------------------------------------------------------
# 3. GENERATE COMPOSITE PANEL FIGURE
# ------------------------------------------------------------------------------
# Using a wide aspect ratio for a two-column journal layout
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(24, 11))
plt.subplots_adjust(wspace=0.35)

# --- PANEL A: Pearson Correlation Heatmap ---
# Note: 'corr' variable is inherited from the previous script cell
sns.heatmap(
    corr_matrix, ax=ax1, cmap='RdBu_r', center=0, annot=True, fmt=".2f",
    annot_kws={'size': 12, 'weight': 'bold', 'color': 'black'},
    linewidths=1.5, linecolor='white',
    cbar_kws={"shrink": 0.8, "label": "Pearson Coefficient (r)"}
)

ax1.set_title('(A) Inter-Variable Correlation Matrix', fontsize=22, pad=25, loc='left')
ax1.tick_params(axis='both', which='major', labelsize=14)

# --- PANEL B: VIF Bar Plot ---
# Note: 'vif_results' is inherited from the previous script cell
# Sort for visual hierarchy
vif_sorted = vif_results.sort_values('VIF', ascending=True)
vif_colors = sns.color_palette("magma", len(vif_sorted))

ax2.barh(vif_sorted['Variable'], vif_sorted['VIF'],
         color=vif_colors, edgecolor='black', linewidth=2.0)

# Add Academic Standard Threshold Line (VIF = 5)
ax2.axvline(5, color='#e74c3c', linestyle='--', linewidth=4, label='Tolerance Limit (VIF = 5)')

ax2.set_title('(B) Variance Inflation Factor (VIF)', fontsize=22, pad=25, loc='left')
ax2.set_xlabel('VIF Value', fontsize=16, labelpad=15)
ax2.grid(axis='x', linestyle=':', alpha=0.6, linewidth=1.5)
ax2.legend(loc='lower right', fontsize=15, frameon=True, shadow=True, borderpad=1)
ax2.tick_params(axis='both', which='major', labelsize=14)

# ------------------------------------------------------------------------------
# 4. FINAL EXPORT (1200 DPI LOSSLESS)
# ------------------------------------------------------------------------------
sns.despine(ax=ax2, left=False)
plt.tight_layout()

# Save Figure 6 to both Output folders
fig_name = 'Figure_6_Multicollinearity_Panel_1200DPI.png'

for folder in [OUT_IDEMILI, OUT_AWKA]:
    save_path = os.path.join(folder, fig_name)
    plt.savefig(save_path, dpi=1200, bbox_inches='tight')

print(f"✅ SUCCESS: Panel Figure 6 (A & B) saved to Outputs/ at 1200 DPI.")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.4 – PRINCIPAL COMPONENT ANALYSIS (PCA)
# 1200 DPI | BOLD BIPLOT | AUTOMATIC REPOSITORY MAPPING | MANUSCRIPT PANEL 7
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Outputs
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

for folder in [OUT_IDEMILI, OUT_AWKA]:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. DATA PREPARATION & STANDARDIZATION
# ------------------------------------------------------------------------------
print("--- Initializing PCA: Dimensionality Reduction ---")
df_idm = pd.read_csv(PATH_IDEMILI)
df_awk = pd.read_csv(PATH_AWKA)

# Standard Cleaning
soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
vars_used = topo_vars + soil_vars

# Remove artifacts and merge
df_idm['Region'] = 'Idemili'
df_awk['Region'] = 'Awka'
df_full = pd.concat([df_idm, df_awk], ignore_index=True)

for col in soil_vars:
    df_full.loc[df_full[col] < 0, col] = np.nan
df_clean = df_full.dropna(subset=vars_used)

# PCA requires standardization (mean=0, variance=1)
X = df_clean[vars_used]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ------------------------------------------------------------------------------
# 3. PCA COMPUTATION & TABLE EXPORT
# ------------------------------------------------------------------------------
pca = PCA(n_components=len(vars_used))
X_pca = pca.fit_transform(X_scaled)

# Table 4.4a: Explained Variance (Scree Data)
eigen_table = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(pca.explained_variance_))],
    'Eigenvalue': pca.explained_variance_.round(3),
    'Variance_Explained(%)': (pca.explained_variance_ratio_ * 100).round(2),
    'Cumulative_Variance(%)': (np.cumsum(pca.explained_variance_ratio_) * 100).round(2)
})

# Table 4.4b: Factor Loadings (Variable Contributions)
loadings_table = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(pca.components_))],
    index=vars_used
).round(3)

# Save Tables
for folder in [OUT_IDEMILI, OUT_AWKA]:
    eigen_table.to_csv(os.path.join(folder, 'Table_4_4a_PCA_Variance.csv'), index=False)
    loadings_table.to_csv(os.path.join(folder, 'Table_4_4b_PCA_Loadings.csv'))

# ------------------------------------------------------------------------------
# 4. PANEL FIGURE 7: SCREE PLOT & BIPLOT (1200 DPI)
# ------------------------------------------------------------------------------
plt.rcParams.update({
    'font.weight': 'bold', 'axes.labelweight': 'bold',
    'axes.linewidth': 2.5, 'font.family': 'sans-serif'
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
plt.subplots_adjust(wspace=0.3)

# --- (A) SCREE PLOT ---
ax1.bar(range(1, len(pca.explained_variance_ratio_)+1),
        pca.explained_variance_ratio_*100, color='#34495e', alpha=0.8, edgecolor='black', label='Individual PC')
ax1.plot(range(1, len(pca.explained_variance_ratio_)+1),
         np.cumsum(pca.explained_variance_ratio_)*100, marker='o', color='#e74c3c', markersize=10, linewidth=4, label='Cumulative %')

ax1.set_title('(A) Scree Plot: Explained Variance', fontsize=20, loc='left', pad=20)
ax1.set_xlabel('Principal Component Number', fontsize=14)
ax1.set_ylabel('Variance Explained (%)', fontsize=14)
ax1.set_ylim(0, 105)
ax1.legend(fontsize=12, frameon=True, shadow=True)
ax1.grid(axis='y', linestyle='--', alpha=0.5, linewidth=1.5)

# --- (B) PCA BIPLOT ---
pc_df = pd.DataFrame(X_pca, columns=[f'PC{i+1}' for i in range(X_pca.shape[1])])
pc_df['Region'] = df_clean['Region'].values

# Regional clusters
sns.scatterplot(data=pc_df, x='PC1', y='PC2', hue='Region',
                palette={'Idemili': '#d35400', 'Awka': '#2c3e50'},
                s=100, alpha=0.5, ax=ax2, edgecolor='white', linewidth=0.8)

# Vector Overlays (Variable Influences)
# Increased scale_factor for high-contrast visual separation
scale_factor = 5
for i, var in enumerate(vars_used):
    dx = pca.components_[0, i] * scale_factor
    dy = pca.components_[1, i] * scale_factor
    ax2.arrow(0, 0, dx, dy, color='black', alpha=0.9, width=0.03, head_width=0.15, linewidth=2)
    ax2.text(dx * 1.15, dy * 1.15, var, color='black', fontsize=14, fontweight='bold', ha='center', va='center')

ax2.set_title('(B) PCA Biplot: Feature Influence (PC1 vs PC2)', fontsize=20, loc='left', pad=20)
ax2.set_xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=14)
ax2.set_ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=14)
ax2.axhline(0, color='grey', linestyle='--', alpha=0.6, linewidth=1.5)
ax2.axvline(0, color='grey', linestyle='--', alpha=0.6, linewidth=1.5)

# ------------------------------------------------------------------------------
# 5. FINAL EXPORT
# ------------------------------------------------------------------------------
sns.despine()
plt.tight_layout()

# Save at 1200 DPI for high-quality printing
fig_name = 'Figure_7_PCA_Analysis_Panel_1200DPI.png'
for folder in [OUT_IDEMILI, OUT_AWKA]:
    plt.savefig(os.path.join(folder, fig_name), dpi=1200, bbox_inches='tight')

print(f"\n✅ SUCCESS: PCA Variance Table and Loadings Table saved.")
print(f"✅ SUCCESS: Panel Figure 7 (A & B) saved at 1200 DPI.")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.5 – BINARY LOGISTIC REGRESSION (BLR) MODEL
# 1200 DPI | ODDS RATIOS | AUC-ROC | REPOSITORY-AWARE FOLDER ORG
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             classification_report, accuracy_score)

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Output Folders
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

for folder in [OUT_IDEMILI, OUT_AWKA]:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. DATA PREPARATION
# ------------------------------------------------------------------------------
print("--- Initializing Binary Logistic Regression ---")
df_idm = pd.read_csv(PATH_IDEMILI)
df_awk = pd.read_csv(PATH_AWKA)
df_full = pd.concat([df_idm, df_awk], ignore_index=True)

# Robust Cleaning
soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
predictors = topo_vars + soil_vars

for col in soil_vars:
    df_full.loc[df_full[col] < 0, col] = np.nan
df_clean = df_full.dropna(subset=predictors + ['Presence']).reset_index(drop=True)

X = df_clean[predictors]
y = df_clean['Presence'].astype(int)

# ------------------------------------------------------------------------------
# 3. MODEL TRAINING: STRATIFIED SPLIT & SCALING
# ------------------------------------------------------------------------------
# 70/30 Stratified Split to maintain class balance in small datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------------------------
# 4. STATISTICAL INFERENCE (Statsmodels)
# ------------------------------------------------------------------------------
# Adding constant for Intercept
X_train_const = sm.add_constant(X_train_scaled)
logit_inf = sm.Logit(y_train, X_train_const).fit(disp=False)

# Calculate Odds Ratios: exp(coeff)
logit_summary = pd.DataFrame({
    'Coefficient': logit_inf.params,
    'Std_Error': logit_inf.bse,
    'Z_Value': logit_inf.tvalues,
    'P_Value': logit_inf.pvalues,
    'Odds_Ratio': np.exp(logit_inf.params)
})
logit_summary.index = ['Intercept'] + predictors

# ------------------------------------------------------------------------------
# 5. MACHINE LEARNING EVALUATION (Scikit-Learn)
# ------------------------------------------------------------------------------
blr_model = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42)
blr_model.fit(X_train_scaled, y_train)

y_pred = blr_model.predict(X_test_scaled)
y_prob = blr_model.predict_proba(X_test_scaled)[:, 1]
auc_val = roc_auc_score(y_test, y_prob)
conf_mat = confusion_matrix(y_test, y_pred)

# ------------------------------------------------------------------------------
# 6. PANEL FIGURE 8: PERFORMANCE DIAGNOSTICS (1200 DPI)
# ------------------------------------------------------------------------------
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 9))
plt.subplots_adjust(wspace=0.3)

# --- (A) ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax1.plot(fpr, tpr, color='#2c3e50', linewidth=4, label=f'BLR AUC = {auc_val:.3f}')
ax1.plot([0, 1], [0, 1], color='#e74c3c', linestyle='--', linewidth=2.5)
ax1.set_title('(A) ROC Curve: Logistic Regression Performance', fontsize=18, loc='left', pad=20)
ax1.set_xlabel('False Positive Rate', fontsize=14)
ax1.set_ylabel('True Positive Rate', fontsize=14)
ax1.legend(loc='lower right', fontsize=15, frameon=True, shadow=True)
ax1.grid(alpha=0.3, linestyle=':')

# --- (B) Confusion Matrix ---
sns.heatmap(conf_mat, annot=True, fmt='d', cmap='RdPu', ax=ax2,
            annot_kws={'size': 20, 'weight': 'bold'}, cbar=False,
            linewidths=2, linecolor='white')
ax2.set_title('(B) Confusion Matrix (Test Set)', fontsize=18, loc='left', pad=20)
ax2.set_xlabel('Predicted Label', fontsize=14)
ax2.set_ylabel('Actual Label', fontsize=14)
ax2.set_xticklabels(['Absence', 'Presence'], fontsize=12)
ax2.set_yticklabels(['Absence', 'Presence'], fontsize=12)

# ------------------------------------------------------------------------------
# 7. EXPORT RESULTS
# ------------------------------------------------------------------------------
plt.tight_layout()
fig_name = 'Figure_8_BLR_Performance_1200DPI.png'
table_name = 'Table_4_5_BLR_Coefficients_Odds.csv'

for folder in [OUT_IDEMILI, OUT_AWKA]:
    plt.savefig(os.path.join(folder, fig_name), dpi=1200, bbox_inches='tight')
    logit_summary.to_csv(os.path.join(folder, table_name))

print(f"\n✅ SUCCESS: BLR Coefficients and Odds Ratios saved.")
print(f"✅ SUCCESS: Panel Figure 8 saved at 1200 DPI (AUC: {auc_val:.4f}).")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.6 – RANDOM FOREST CLASSIFICATION (RF)
# 1200 DPI | FEATURE IMPORTANCE | BALANCED EVALUATION | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             classification_report, accuracy_score)

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Output Folders
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

for folder in [OUT_IDEMILI, OUT_AWKA]:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. DATA PREPARATION
# ------------------------------------------------------------------------------
print("--- Initializing Random Forest Modeling Pipeline ---")
df_idm = pd.read_csv(PATH_IDEMILI)
df_awk = pd.read_csv(PATH_AWKA)
df_full = pd.concat([df_idm, df_awk], ignore_index=True)

# Variables and Cleaning
soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
predictors = topo_vars + soil_vars

for col in soil_vars:
    df_full.loc[df_full[col] < 0, col] = np.nan
df_clean = df_full.dropna(subset=predictors + ['Presence']).reset_index(drop=True)

X = df_clean[predictors]
y = df_clean['Presence'].astype(int)

# ------------------------------------------------------------------------------
# 3. STRATIFIED DATA SPLIT
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# ------------------------------------------------------------------------------
# 4. RANDOM FOREST TRAINING (500 TREES)
# ------------------------------------------------------------------------------
# Using 'balanced' class weights to handle sample size variations
rf_model = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    bootstrap=True
)
rf_model.fit(X_train, y_train)

# ------------------------------------------------------------------------------
# 5. PERFORMANCE METRICS
# ------------------------------------------------------------------------------
y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:, 1]
auc_val = roc_auc_score(y_test, y_prob)

# Feature Importance extraction
importances = pd.Series(
    rf_model.feature_importances_,
    index=predictors
).sort_values(ascending=True)

# ------------------------------------------------------------------------------
# 6. PANEL FIGURE 9: IMPORTANCE & ROC CURVE (1200 DPI)
# ------------------------------------------------------------------------------
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2.5})
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
plt.subplots_adjust(wspace=0.35)

# --- (A) Variable Importance Plot ---
ax1.barh(importances.index, importances.values, color='#2c3e50', edgecolor='black', linewidth=1.5)
ax1.set_title('(A) Random Forest Feature Importance', fontsize=20, loc='left', pad=25)
ax1.set_xlabel('Mean Decrease in Gini Impurity (Contribution)', fontsize=16, labelpad=15)
ax1.grid(axis='x', linestyle=':', alpha=0.5, linewidth=1.5)
ax1.tick_params(axis='both', labelsize=14)

# --- (B) ROC Curve ---
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#d35400', linewidth=5, label=f'RF AUC = {auc_val:.3f}')
ax2.plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=2.5, alpha=0.7)
ax2.set_title('(B) ROC Curve: Model Predictive Accuracy', fontsize=20, loc='left', pad=25)
ax2.set_xlabel('False Positive Rate', fontsize=16, labelpad=15)
ax2.set_ylabel('True Positive Rate', fontsize=16, labelpad=15)
ax2.legend(loc='lower right', fontsize=16, frameon=True, shadow=True, borderpad=1)
ax2.grid(alpha=0.3, linestyle=':')
ax2.tick_params(axis='both', labelsize=14)

# ------------------------------------------------------------------------------
# 7. EXPORT DATA & VISUALS
# ------------------------------------------------------------------------------
plt.tight_layout()
fig_name = 'Figure_9_RF_Performance_Panel_1200DPI.png'
importance_csv = 'Table_4_6_RF_Feature_Importance.csv'
predictions_csv = 'Table_4_7_RF_Validation_Test_Set.csv'

# Validation data for manual checking/tables
test_output = X_test.copy()
test_output['Actual_Presence'] = y_test
test_output['Predicted_Presence'] = y_pred
test_output['Gully_Probability'] = y_prob

for folder in [OUT_IDEMILI, OUT_AWKA]:
    plt.savefig(os.path.join(folder, fig_name), dpi=1200, bbox_inches='tight')
    importances.to_csv(os.path.join(folder, importance_csv))
    test_output.to_csv(os.path.join(folder, predictions_csv), index=False)

print(f"\n✅ SUCCESS: Random Forest AUC: {auc_val:.4f}")
print(f"✅ SUCCESS: Feature Importance and Test Set Predictions exported.")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.7 – FINAL MODEL COMPARISON (BLR VS. RF)
# 1200 DPI | ROC COMPARISON | COMPREHENSIVE METRICS | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (roc_curve, roc_auc_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix)

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Output Folders
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

for folder in [OUT_IDEMILI, OUT_AWKA]:
    os.makedirs(folder, exist_ok=True)

# ------------------------------------------------------------------------------
# 2. DATA PREPARATION & CLEANING
# ------------------------------------------------------------------------------
print("--- Loading and Merging Data for Final Comparison ---")
df_idm = pd.read_csv(PATH_IDEMILI)
df_awk = pd.read_csv(PATH_AWKA)
df_full = pd.concat([df_idm, df_awk], ignore_index=True)

soil_vars = ['Sand', 'Clay', 'SOC', 'BulkDensity', 'CEC', 'pH']
topo_vars = ['Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI']
predictors = topo_vars + soil_vars

# Clean SoilGrids artifacts
for c in soil_vars:
    df_full.loc[df_full[c] < 0, c] = np.nan
df_clean = df_full.dropna(subset=predictors + ['Presence']).reset_index(drop=True)

X = df_clean[predictors]
y = df_clean['Presence'].astype(int)

# ------------------------------------------------------------------------------
# 3. STRATIFIED SPLIT & SCALING
# ------------------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Standardize specifically for Logistic Regression convergence
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ------------------------------------------------------------------------------
# 4. SIMULTANEOUS MODEL TRAINING
# ------------------------------------------------------------------------------
print("--- Training BLR and RF Models ---")

# Logistic Regression (On Scaled Data)
blr_model = LogisticRegression(max_iter=5000, class_weight='balanced', solver='lbfgs', random_state=42)
blr_model.fit(X_train_scaled, y_train)
y_prob_blr = blr_model.predict_proba(X_test_scaled)[:, 1]
y_pred_blr = blr_model.predict(X_test_scaled)

# Random Forest (On Original Data)
rf_model = RandomForestClassifier(n_estimators=500, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]
y_pred_rf = rf_model.predict(X_test)

# ------------------------------------------------------------------------------
# 5. STATISTICAL PERFORMANCE SUMMARY
# ------------------------------------------------------------------------------
def get_metrics(true, pred, prob, name):
    return {
        'Model': name,
        'Accuracy': accuracy_score(true, pred),
        'Precision': precision_score(true, pred),
        'Recall (Sensitivity)': recall_score(true, pred),
        'F1_Score': f1_score(true, pred),
        'AUC': roc_auc_score(true, prob)
    }

comparison_table = pd.DataFrame([
    get_metrics(y_test, y_pred_blr, y_prob_blr, 'Logistic Regression (BLR)'),
    get_metrics(y_test, y_pred_rf, y_prob_rf, 'Random Forest (RF)')
]).round(4)

print("\n--- FINAL MODEL PERFORMANCE COMPARISON ---")
print(comparison_table)

# ------------------------------------------------------------------------------
# 6. PANEL FIGURE 10: HEAD-TO-HEAD COMPARISON (1200 DPI)
# ------------------------------------------------------------------------------
plt.rcParams.update({
    'font.weight': 'bold', 'axes.labelweight': 'bold',
    'axes.linewidth': 2.5, 'font.family': 'sans-serif'
})

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 10))
plt.subplots_adjust(wspace=0.35)

# --- (A) Combined ROC Curve Comparison ---
fpr_blr, tpr_blr, _ = roc_curve(y_test, y_prob_blr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

ax1.plot(fpr_blr, tpr_blr, color='#34495e', linewidth=4, alpha=0.9,
         label=f'BLR Model (AUC={comparison_table.iloc[0]["AUC"]:.3f})')
ax1.plot(fpr_rf, tpr_rf, color='#d35400', linewidth=5,
         label=f'RF Model (AUC={comparison_table.iloc[1]["AUC"]:.3f})')
ax1.plot([0, 1], [0, 1], color='black', linestyle='--', linewidth=2.5, alpha=0.6)

ax1.set_title('(A) ROC CURVE: MODEL ACCURACY COMPARISON', loc='left', fontsize=20, pad=25)
ax1.set_xlabel('FALSE POSITIVE RATE', fontsize=15, labelpad=15)
ax1.set_ylabel('TRUE POSITIVE RATE (RECALL)', fontsize=15, labelpad=15)
ax1.legend(frameon=True, loc='lower right', fontsize=14, shadow=True, borderpad=1)
ax1.grid(True, linestyle=':', alpha=0.5, linewidth=1.5)

# --- (B) RF Confusion Matrix (Top Performing Model) ---
cm_rf = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='YlOrBr', ax=ax2,
            annot_kws={"size": 24, "weight": "bold"}, cbar=False,
            linewidths=3, linecolor='white')

ax2.set_title('(B) CONFUSION MATRIX: RANDOM FOREST (TEST SET)', loc='left', fontsize=20, pad=25)
ax2.set_xticklabels(['ABSENCE', 'PRESENCE'], fontsize=14, fontweight='bold')
ax2.set_yticklabels(['ABSENCE', 'PRESENCE'], fontsize=14, fontweight='bold')
ax2.set_xlabel('PREDICTED CATEGORY', fontsize=15, labelpad=15)
ax2.set_ylabel('ACTUAL CATEGORY', fontsize=15, labelpad=15)

# ------------------------------------------------------------------------------
# 7. FINAL EXPORTS (PUBLICATION READY)
# ------------------------------------------------------------------------------
plt.tight_layout()
fig_name = 'Figure_10_Final_Model_Comparison_1200DPI.png'
table_name = 'Table_4_8_Performance_Metrics_Final.csv'

for folder in [OUT_IDEMILI, OUT_AWKA]:
    plt.savefig(os.path.join(folder, fig_name), dpi=1200, bbox_inches='tight')
    comparison_table.to_csv(os.path.join(folder, table_name), index=False)

print(f"\n✅ SUCCESS: Final Performance Comparison saved to Outputs/.")
print(f"✅ STATUS: Use Table_4_8 for the results section of your manuscript.")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.8A – GEOSPATIAL COVARIATE VISUALIZATION (IDEMILI ONLY)
# 1200 DPI | RASTER MASKING | MULTI-PANEL ATLAS | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import os
import rasterio
import rasterio.mask
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (STRICTLY IDEMILI)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())

# Inputs
SHP_PATH = os.path.join(BASE_DIR, 'Shapefile', 'IDEMILI_NS', 'IDEMILI_NS.shp')
TIFF_DIR = os.path.join(BASE_DIR, 'Data', 'Raw_GeoTIFFs', 'IDEMILI')

# Output
OUT_FOLDER = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
if not os.path.exists(OUT_FOLDER):
    os.makedirs(OUT_FOLDER)

# Define the specific 12 covariates for the panel
covariates = [
    'Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI', 'SPI',
    'Hillshade', 'Sand', 'Clay', 'SOC', 'BulkDensity', 'pH'
]

# ------------------------------------------------------------------------------
# 2. LOAD BOUNDARY & INITIALIZE PLOT
# ------------------------------------------------------------------------------
print(f"--- Processing Idemili Raster Masking ---")
boundary = gpd.read_file(SHP_PATH)

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2.5})
fig, axes = plt.subplots(4, 3, figsize=(20, 24), facecolor='white')
axes = axes.flatten()

# ------------------------------------------------------------------------------
# 3. MASKING & MAPPING LOOP
# ------------------------------------------------------------------------------
for i, cov in enumerate(covariates):
    file_name = f"IDEMILI_{cov}.tif"
    full_path = os.path.join(TIFF_DIR, file_name)
    ax = axes[i]

    if not os.path.exists(full_path):
        print(f"⚠️ Warning: {file_name} not found. Skipping axis.")
        ax.axis('off')
        continue

    with rasterio.open(full_path) as src:
        # Match Coordinate Reference System (CRS)
        boundary_proj = boundary.to_crs(src.crs)

        # Apply Mask (Crop to Boundary)
        out_image, out_transform = rasterio.mask.mask(src, boundary_proj.geometry, crop=True)

        # Convert to Float32 to allow for NaN transparency
        out_data = out_image[0].astype('float32')

        # Set NoData and Outliers to NaN
        nodata_val = src.nodata
        out_data[out_data == nodata_val] = np.nan
        out_data[out_data < -9000] = np.nan

        # Assign Stylized Colormaps
        if any(x in cov for x in ['Slope', 'Elevation']):
            cmap = 'terrain'
        elif any(x in cov for x in ['TWI', 'SPI']):
            cmap = 'GnBu'
        elif any(x in cov for x in ['pH', 'SOC', 'Sand', 'Clay']):
            cmap = 'YlOrBr'
        elif 'Hillshade' in cov:
            cmap = 'gray'
        else:
            cmap = 'viridis'

        # Render Raster
        im = ax.imshow(out_data, cmap=cmap, interpolation='bilinear')

        # Formatting Axis and Borders
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor('black')

        # Add High-Contrast Colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=10, width=2)

        # Labeling
        ax.set_title(f"IDEMILI: {cov.upper()}", fontsize=16, fontweight='bold', pad=15)

# ------------------------------------------------------------------------------
# 4. FINAL EXPORT
# ------------------------------------------------------------------------------
plt.tight_layout(pad=5.0)
save_path = os.path.join(OUT_FOLDER, 'Figure_11_Idemili_Covariate_Atlas_1200DPI.png')

plt.savefig(save_path, dpi=1200, bbox_inches='tight')
print(f"\n✅ SUCCESS: Idemili Atlas saved to {save_path}")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.8B – GEOSPATIAL COVARIATE VISUALIZATION (AWKA ONLY)
# 1200 DPI | RASTER MASKING | MULTI-PANEL ATLAS | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import os
import rasterio
import rasterio.mask
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (STRICTLY AWKA)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())

# Inputs
SHP_PATH_AWKA = os.path.join(BASE_DIR, 'Shapefile', 'AWKA_NS', 'AWKA_NS.shp')
TIFF_DIR_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_GeoTIFFs', 'AWKA')

# Output
OUT_FOLDER_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')
if not os.path.exists(OUT_FOLDER_AWKA):
    os.makedirs(OUT_FOLDER_AWKA)

# Standardized 12-file list for Awka
covariates = [
    'Elevation', 'Slope', 'Aspect', 'Curvature', 'TWI', 'SPI',
    'Hillshade', 'Sand', 'Clay', 'SOC', 'BulkDensity', 'pH'
]

# ------------------------------------------------------------------------------
# 2. LOAD BOUNDARY & INITIALIZE PLOT
# ------------------------------------------------------------------------------
print(f"--- Processing Awka Raster Masking ---")
boundary_awka = gpd.read_file(SHP_PATH_AWKA)

plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold', 'axes.linewidth': 2.5})
fig, axes = plt.subplots(4, 3, figsize=(20, 24), facecolor='white')
axes = axes.flatten()

# ------------------------------------------------------------------------------
# 3. MASKING & MAPPING LOOP
# ------------------------------------------------------------------------------
for i, cov in enumerate(covariates):
    file_name = f"AWKA_{cov}.tif"
    full_path = os.path.join(TIFF_DIR_AWKA, file_name)
    ax = axes[i]

    if not os.path.exists(full_path):
        print(f"⚠️ Warning: {file_name} not found in {TIFF_DIR_AWKA}")
        ax.axis('off')
        continue

    with rasterio.open(full_path) as src:
        # Align Coordinate Reference Systems
        boundary_proj = boundary_awka.to_crs(src.crs)

        # Mask/Clip to Awka Boundary
        out_image, out_transform = rasterio.mask.mask(src, boundary_proj.geometry, crop=True)

        # Convert to Float32 for NaN handling (White background)
        out_data = out_image[0].astype('float32')

        # Set NoData values and extreme artifacts to NaN
        nodata_val = src.nodata
        out_data[out_data == nodata_val] = np.nan
        out_data[out_data < -500] = np.nan

        # Unified Colormap Logic
        if any(x in cov for x in ['Slope', 'Elevation']):
            cmap = 'terrain'
        elif any(x in cov for x in ['TWI', 'SPI']):
            cmap = 'GnBu'
        elif any(x in cov for x in ['pH', 'SOC', 'Sand', 'Clay']):
            cmap = 'YlOrBr'
        elif 'Hillshade' in cov:
            cmap = 'gray'
        else:
            cmap = 'viridis'

        # Render Raster Plot
        im = ax.imshow(out_data, cmap=cmap, interpolation='bilinear')

        # Bold Styling and Frames
        ax.set_facecolor('white')
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor('black')

        # Individual Scale Bar (Colorbar)
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(im, cax=cax)
        cbar.ax.tick_params(labelsize=10, width=2)

        # Title formatting
        ax.set_title(f"AWKA: {cov.upper()}", fontsize=16, fontweight='bold', pad=15)

# ------------------------------------------------------------------------------
# 4. FINAL EXPORT (MANUSCRIPT READY)
# ------------------------------------------------------------------------------
plt.tight_layout(pad=5.0)
save_path_awka = os.path.join(OUT_FOLDER_AWKA, 'Figure_12_Awka_Covariate_Atlas_1200DPI.png')

plt.savefig(save_path_awka, dpi=1200, bbox_inches='tight')
print(f"\n✅ SUCCESS: Awka Atlas saved to {save_path_awka}")
plt.show()

In [ ]:
# ==============================================================================
# SECTION 4.9 – OBJECTIVE WEIGHT EXTRACTION (RF GINI IMPORTANCE)
# 1200 DPI | ARCGIS-READY INFLUENCE % | REPOSITORY-AWARE
# PRIMARY AUTHOR: Nelson Onyebuchi Nwobi
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.ensemble import RandomForestClassifier

# ------------------------------------------------------------------------------
# 1. PATH CONFIGURATION (GITHUB STRUCTURE)
# ------------------------------------------------------------------------------
BASE_DIR = os.path.dirname(os.getcwd())
PATH_AWKA = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Awka_23_v2.csv')
PATH_IDEMILI = os.path.join(BASE_DIR, 'Data', 'Raw_CSV', 'Balanced_Samples_Idemili_162_v2.csv')

# Outputs
OUT_IDEMILI = os.path.join(BASE_DIR, 'Outputs', 'IDEMILI')
OUT_AWKA = os.path.join(BASE_DIR, 'Outputs', 'AWKA')

# ------------------------------------------------------------------------------
# 2. DATA LOAD & PRE-PROCESSING
# ------------------------------------------------------------------------------
df_awk = pd.read_csv(PATH_AWKA)
df_idm = pd.read_csv(PATH_IDEMILI)
df = pd.concat([df_awk, df_idm], ignore_index=True)

# Isolate Predictors: Remove all GIS metadata and the target variable
target = 'Presence'
cols_to_drop = [target, 'system:index', '.geo', 'id', 'Unnamed: 0', 'Region', 'X', 'Y']
X = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
y = df[target]

# Handle SoilGrids NoData (-9999) and replace with mean for RF stability
X = X.apply(pd.to_numeric, errors='coerce')
X[X < -500] = np.nan
X = X.fillna(X.mean())

print(f"✅ Training RF on {len(X.columns)} factors across {len(df)} samples...")

# ------------------------------------------------------------------------------
# 3. EXTRACT GINI IMPORTANCE (500 TREES)
# ------------------------------------------------------------------------------
rf_final = RandomForestClassifier(n_estimators=500, random_state=42, n_jobs=-1)
rf_final.fit(X, y)

# 4. CALCULATE ARCGIS INFLUENCE PERCENTAGES
importance_df = pd.DataFrame({
    'Predictor': X.columns,
    'Gini_Weight': rf_final.feature_importances_
}).sort_values(by='Gini_Weight', ascending=False)

# Convert to whole numbers for ArcGIS Weighted Overlay
importance_df['Influence_Percentage'] = (importance_df['Gini_Weight'] * 100).round(0).astype(int)

# --- CRITICAL FIX: Ensure Total is exactly 100% ---
current_total = importance_df['Influence_Percentage'].sum()
if current_total != 100:
    diff = 100 - current_total
    # Adjust the top predictor to make the sum exactly 100
    importance_df.iloc[0, 2] += diff

# ------------------------------------------------------------------------------
# 5. VISUALIZATION & EXPORT
# ------------------------------------------------------------------------------
plt.rcParams.update({'font.weight': 'bold', 'axes.labelweight': 'bold'})
plt.figure(figsize=(12, 7))

# Publication-style Bar Chart
colors = plt.cm.magma(np.linspace(0.3, 0.8, len(importance_df)))
plt.barh(importance_df['Predictor'], importance_df['Influence_Percentage'],
         color=colors, edgecolor='black', linewidth=1.5)

plt.xlabel('ArcGIS Weighted Overlay Influence (%)', fontsize=12)
plt.title('Figure 13: Objective Weights for Gully Vulnerability Mapping', fontsize=16, pad=20)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.6)

# Save results
for folder in [OUT_IDEMILI, OUT_AWKA]:
    importance_df.to_csv(os.path.join(folder, 'Table_4_9_ArcGIS_Weights.csv'), index=False)
    plt.savefig(os.path.join(folder, 'Figure_13_Objective_Weights_1200DPI.png'), dpi=1200, bbox_inches='tight')

# Display final table
print("\n" + "="*50)
print("   FINAL WEIGHTS FOR ARCGIS OVERLAY TOOL")
print("="*50)
print(importance_df[['Predictor', 'Influence_Percentage']].to_string(index=False))
print("="*50)
print(f"Total: {importance_df['Influence_Percentage'].sum()}%")

plt.show()